In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

In [2]:
class NERHead(nn.Module):
    """
    Named Entity Recognition (NER) head for identifying specific entity types.
    """
    def __init__(self, hidden_size, num_labels):
        super(NERHead, self).__init__()
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, sequence_output):
        """
        Forward pass for token classification.
        - sequence_output: (batch, seq_len, hidden_size)
        Returns:
        - Token classification logits: (batch, seq_len, num_labels)
        """
        return self.classifier(sequence_output)  # Token classification per entity type



In [3]:
class CrossAttentionLayer(nn.Module):
    """
    Cross Attention mechanism that computes relationships between Personal NER and PII tokens.
    Uses Personal NER as Query, PII NER as Key/Value.
    """
    def __init__(self, hidden_size):
        super(CrossAttentionLayer, self).__init__()
        self.query_dense = nn.Linear(hidden_size, hidden_size)  # Query (Personal entities)
        self.key_dense = nn.Linear(hidden_size, hidden_size)    # Key (PII entities)
        self.value_dense = nn.Linear(hidden_size, hidden_size)  # Value (PII entities)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequence_output, personal_logits, pii_logits):
        """
        Inputs:
        - sequence_output: (batch, seq_len, hidden_size) - Raw BERT embeddings
        - personal_logits: (batch, seq_len, num_labels) - PERSONAL NER classification logits
        - pii_logits: (batch, seq_len, num_labels) - PII NER classification logits
        
        Returns:
        - attention_output: Weighted representation of PII entities for each Personal entity
        - attention_weights: Attention distribution over PII tokens
        """
        batch_size, seq_len, hidden_dim = sequence_output.shape

        # Compute softmax probabilities over entity labels
        personal_probs = F.softmax(personal_logits, dim=-1).sum(dim=-1, keepdim=True)  # (batch, seq_len, 1)
        pii_probs = F.softmax(pii_logits, dim=-1).sum(dim=-1, keepdim=True)  # (batch, seq_len, 1)

        # Project sequence embeddings into QKV space
        queries = self.query_dense(sequence_output) * personal_probs  # Personal entities as queries
        keys = self.key_dense(sequence_output) * pii_probs  # PII entities as keys
        values = self.value_dense(sequence_output) * pii_probs  # PII entities as values

        # Compute scaled dot-product attention
        scores = torch.matmul(queries, keys.transpose(-2, -1)) / (hidden_dim ** 0.5)  # (batch, seq_len, seq_len)
        attention_weights = self.softmax(scores)  # Normalize across the sequence

        # Compute weighted sum of values
        attention_output = torch.matmul(attention_weights, values)  # (batch, seq_len, hidden_size)

        return attention_output, attention_weights



In [4]:
class PrivacyClassificationHead(nn.Module):
    """
    Final classification head that determines if a given document contains privacy-sensitive information.
    """
    def __init__(self, hidden_size, num_classes=2):
        super(PrivacyClassificationHead, self).__init__()
        self.fc = nn.Linear(hidden_size, num_classes)  # Binary classification (Sensitive / Not Sensitive)

    def forward(self, cross_attention_output):
        """
        Inputs:
        - cross_attention_output: (batch, seq_len, hidden_size) - Processed representation after cross-attention
        
        Returns:
        - Privacy classification logits: (batch, num_classes)
        """
        pooled_output = torch.mean(cross_attention_output, dim=1)  # Pool sequence-level representation
        return self.fc(pooled_output)  # Binary classification (Sensitive / Not Sensitive)



In [5]:
class PrivacyDetectionModel(nn.Module):
    """
    Full Privacy Detection Model with:
    - ModernBERT as backbone
    - Two independent NER heads (Personal, PII)
    - Cross Attention between NER outputs
    - Privacy classification based on attention output
    """
    def __init__(self, base_model, num_labels_personal, num_labels_pii):
        """
        - base_model: Pretrained ModernBERT backbone
        - num_labels_personal: Number of labels for PERSONAL NER
        - num_labels_pii: Number of labels for PII NER
        """
        super(PrivacyDetectionModel, self).__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size

        # Independent NER heads
        self.personal_ner_head = NERHead(hidden_size, num_labels_personal)
        self.pii_ner_head = NERHead(hidden_size, num_labels_pii)

        # Cross-Attention between PERSONAL and PII
        self.cross_attention = CrossAttentionLayer(hidden_size)

        # Final Privacy Classification
        self.privacy_classifier = PrivacyClassificationHead(hidden_size)

    def forward(self, input_ids, attention_mask):
        """
        Inputs:
        - input_ids: (batch, seq_len)
        - attention_mask: (batch, seq_len)
        
        Returns:
        - personal_ner_logits: (batch, seq_len, num_labels_personal)
        - pii_ner_logits: (batch, seq_len, num_labels_pii)
        - attention_output: (batch, seq_len, hidden_size)
        - privacy_classification_logits: (batch, num_classes)
        """
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # BERT sequence embeddings

        # Step 1: NER Prediction
        personal_ner_logits = self.personal_ner_head(sequence_output)
        pii_ner_logits = self.pii_ner_head(sequence_output)

        # Step 2: Cross Attention
        attention_output, attention_weights = self.cross_attention(sequence_output, personal_ner_logits, pii_ner_logits)

        # Step 3: Privacy Classification
        privacy_classification_logits = self.privacy_classifier(attention_output)

        return {
            "personal_ner_logits": personal_ner_logits,  
            "pii_ner_logits": pii_ner_logits,  
            "attention_output": attention_output,  
            "attention_weights": attention_weights,  
            "privacy_classification_logits": privacy_classification_logits  
        }


In [6]:
model_name = "answerdotai/ModernBERT-base"
base_model = AutoModel.from_pretrained(model_name)
model = PrivacyDetectionModel(base_model, num_labels_personal=5, num_labels_pii=6)
print(model)

PrivacyDetectionModel(
  (base_model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (